# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# Gradient boosting because it is a regression task on a tabular data. 
# Because it learns from its own mistakes in the previous trees, it can identify complex relationships from the data.
# This enables it to predict future clicks, impressions, and positions from past clicks, impressions, and positions, etc.

In [489]:
%pip install datasets
from datasets import load_dataset
ds1 = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")
ds2 = load_dataset("FlyRank/internship-warehouse", "dim_content", streaming=True, split="train")
updated = {}
for i in ds2:
    updated[(i['client_hash_id'], i['content_hash_id'])] = i['content_updated_date']
count = matched = 0
dic = {}
for i in ds1:
    count += 1
    if count % 10**5 == 0: print(count, matched)
    if not i["gsc_data_available"]: continue 
    t = (i['client_hash_id'], i['content_hash_id'])
    if t not in updated or updated[t] > i['report_date']: continue
    if t not in dic: dic[t] = set()
    dic[t].add(i['report_date'])
    matched += 1


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
100000 0
200000 0
300000 0
400000 0
500000 0
600000 0
700000 0
800000 0
900000 0
1000000 0
1100000 1
1200000 2
1300000 98
1400000 222
1500000 336
1600000 442
1700000 490
1800000 856
1900000 1182
2000000 1495
2100000 1753
2200000 1880
2300000 2418
2400000 2644
2500000 2984
2600000 3302
2700000 3393
2800000 3530
2900000 3962
3000000 4597
3100000 4886
3200000 5158
3300000 5370
3400000 5489
3500000 5677
3600000 5762
3700000 5831
3800000 5991
3900000 6245
4000000 6326
4100000 6418
4200000 6658
4300000 6685
4400000 6780
4500000 6884
4600000 7063
4700000 7107
4800000 7131
4900000 7288
5000000 7378
5100000 7484
5200000 7584
5300000 7643
5400000 7850
5500000 7889
5600000 8553
5700000 8752
5800000 9816
5900000 10381
6000000 11613
6100000 11790
6200000 12812
6300000 14492
6400000 15563
6500000 16169
6600000 16

In [490]:
from datetime import timedelta
import random
random.seed(42)
def get_streaks(dates):
    if len(dates) == 0: return []
    dates = sorted(dates)
    streaks = [[dates[0]]]
    for i in range(1, len(dates)):
        if dates[i] == dates[i-1] + timedelta(days=1):
            streaks[-1].append(dates[i])
            if len(streaks[-1]) == 90:
                streaks.append(streaks[-1][15:])
                streaks[-2] = (streaks[-2][0], streaks[-2][-1])
        else: 
            streaks[-1] = [dates[i]]
    if type(streaks[-1]) == list: streaks.pop()
    return streaks
streaks = []
for i in dic:
    s = get_streaks(dic[i])
    if len(s) > 0: streaks.append((i, s))
total = sum(len(streak) for _, streak in streaks)
random.shuffle(streaks)
valindex = testindex = 0
cur = 0
for i in range(len(streaks)):
    cur += len(streaks[i][1])
    if cur >= 0.8 * total and valindex == 0: valindex = i+1
    if cur >= 0.9 * total and testindex == 0: testindex = i+1
train_pages = streaks[:valindex]
val_pages = streaks[valindex:testindex]
test_pages = streaks[testindex:]
page_blocks = {}
data_id = 0
splits = {"train": train_pages, "val": val_pages, "test": test_pages}
for split_name, pages in splits.items():
    for page, blocks in pages:
        page_blocks[page] = {"split": split_name, "blocks": []}
        for start, end in blocks:
            page_blocks[page]["blocks"].append((start, end, data_id))
            data_id += 1

In [491]:
content_info = {}
for i in ds2:
    content_info[(i['client_hash_id'], i['content_hash_id'])] = {'word_count': i['word_count'],
                                                                'update_date': i['content_updated_date'], 
                                                                'create_date': i['content_created_date'], 
                                                                'content_type': i['content_type'],
                                                                'main_intent': i['main_intent']}
data = {'train': {}, 'val': {}, 'test': {}}
for _, i in page_blocks.items():
    for start, end, data_id in i['blocks']:
        data[i['split']][data_id] = {'clicks': [0] * 5, 
                                     'impressions': [0] * 5, 
                                     'position': [0] * 5,
                                     'word_count': None,
                                     'update_date': None,
                                     'create_date': None,
                                     'content_type': None,
                                     'main_intent': None}
count = matched = 0
for i in ds1:
    t = (i['client_hash_id'], i['content_hash_id'])
    count += 1
    if count % 10**5 == 0: print(count, matched)
    if t not in page_blocks: continue
    for start, end, data_id in page_blocks[t]['blocks']:
        if start <= i['report_date'] <= end:
            delta = (i['report_date'] - start).days
            data[page_blocks[t]['split']][data_id]['clicks'][min(4, delta // 15)] += i['gsc_clicks']
            data[page_blocks[t]['split']][data_id]['impressions'][min(4, delta // 15)] += i['gsc_impressions']
            data[page_blocks[t]['split']][data_id]['position'][min(4, delta // 15)] += i['gsc_sum_position']
            data[page_blocks[t]['split']][data_id]['word_count'] = content_info[t]['word_count']
            data[page_blocks[t]['split']][data_id]['update_date'] = (start - content_info[t]['update_date']).days
            data[page_blocks[t]['split']][data_id]['create_date'] = (start - content_info[t]['create_date']).days
            data[page_blocks[t]['split']][data_id]['content_type'] = content_info[t]['content_type']
            data[page_blocks[t]['split']][data_id]['main_intent'] = content_info[t]['main_intent']
            matched += 1
for split in data:
    for i in data[split]:
        for j in range(5):
            if data[split][i]['impressions'][j] == 0: data[split][i]['position'][j] = None
            else: data[split][i]['position'][j] /= data[split][i]['impressions'][j]

100000 0
200000 0
300000 0
400000 0
500000 0
600000 0
700000 0
800000 0
900000 0
1000000 0
1100000 0
1200000 0
1300000 3
1400000 9
1500000 14
1600000 19
1700000 23
1800000 92
1900000 145
2000000 212
2100000 253
2200000 267
2300000 379
2400000 424
2500000 620
2600000 831
2700000 883
2800000 947
2900000 1173
3000000 1475
3100000 1607
3200000 1731
3300000 1942
3400000 2029
3500000 2224
3600000 2294
3700000 2361
3800000 2508
3900000 2785
4000000 2871
4100000 2955
4200000 3304
4300000 3304
4400000 3395
4500000 3486
4600000 3685
4700000 3685
4800000 3685
4900000 3888
5000000 3989
5100000 4094
5200000 4186
5300000 4186
5400000 4499
5500000 4499
5600000 4607
5700000 4826
5800000 4863
5900000 4863
6000000 5011
6100000 5018
6200000 5035
6300000 5108
6400000 5153
6500000 5153
6600000 5153
6700000 5263
6800000 5596
6900000 5596
7000000 5630
7100000 5630
7200000 5630
7300000 5747
7400000 5854
7500000 5865
7600000 5900
7700000 6126
7800000 6344
7900000 6391
8000000 6468
8100000 6468
8200000 6481
830

In [511]:
import numpy as np
X_train, y_train1, y_train2, y_train3 = [], [], [], []
main_intent = {'informational': 0, 'commercial': 1, 'transactional': 2, 'navigational': 3, None: None}
for i in data['train'].values():
    X_train.append(i['clicks'][:4] 
                   + i['impressions'][:4] 
                   + i['position'][:4] 
                   + [i['word_count']]
                   + [i['update_date']] 
                   + [i['create_date']] 
                   + [main_intent[i['main_intent']]])
    y_train1.append(i['clicks'][4])
    y_train2.append(i['impressions'][4])
    y_train3.append(i['position'][4])
X_train, y_train1, y_train2, y_train3 = np.array(X_train), np.array(y_train1), np.array(y_train2), np.array(y_train3)
X_val, y_val1, y_val2, y_val3 = [], [], [], []
for i in data['val'].values():
    X_val.append(i['clicks'][:4] 
                + i['impressions'][:4] 
                + i['position'][:4] 
                + [i['word_count']] 
                + [i['update_date']] 
                + [i['create_date']] 
                + [main_intent[i['main_intent']]])
    y_val1.append(i['clicks'][4])
    y_val2.append(i['impressions'][4])
    y_val3.append(i['position'][4])
X_val, y_val1, y_val2, y_val3 = np.array(X_val), np.array(y_val1), np.array(y_val2), np.array(y_val3)
X_test, y_test1, y_test2, y_test3 = [], [], [], []
for i in data['test'].values():
    X_test.append(i['clicks'][:4] 
                + i['impressions'][:4] 
                + i['position'][:4] 
                + [i['word_count']] 
                + [i['update_date']] 
                + [i['create_date']] 
                + [main_intent[i['main_intent']]])
    y_test1.append(i['clicks'][4])
    y_test2.append(i['impressions'][4])
    y_test3.append(i['position'][4])
X_test, y_test1, y_test2, y_test3 = np.array(X_test), np.array(y_test1), np.array(y_test2), np.array(y_test3)

In [509]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
def test_model(X_train, y_train, X_val, y_val):
    best = [(0, 0, 0, 0), float('-inf')]
    for max_iter in [100, 200, 300, 500, 800]:
        for learning_rate in [0.01, 0.025, 0.05, 0.1]:
            for min_samples_leaf in [10, 20, 30, 50]:
                for l2_regularization in [0, 0.1, 1, 10]:
                    model = HistGradientBoostingRegressor(
                            max_depth=5,
                            max_iter=max_iter,
                            learning_rate=learning_rate,
                            min_samples_leaf=min_samples_leaf,
                            l2_regularization=l2_regularization,
                            early_stopping=False,
                            random_state=42)
                    model.fit(X_train, y_train)
                    pred = model.predict(X_val)
                    r2 = r2_score(y_val, pred)
                    if r2 > best[1]: best = [(max_iter, learning_rate, min_samples_leaf, l2_regularization), r2]
    return best
best1 = test_model(X_train, y_train1, X_val, y_val1)
best2 = test_model(X_train, y_train2, X_val, y_val2)
best3 = test_model(X_train, y_train3, X_val, y_val3)
model1 = HistGradientBoostingRegressor(
    max_depth=5,
    max_iter=best1[0][0],
    learning_rate=best1[0][1],
    min_samples_leaf=best1[0][2],
    l2_regularization=best1[0][3],
    early_stopping=False,
    random_state=42
)
model2 = HistGradientBoostingRegressor(
    max_depth=5,
    max_iter=best2[0][0],
    learning_rate=best2[0][1],
    min_samples_leaf=best2[0][2],
    l2_regularization=best2[0][3],
    early_stopping=False,
    random_state=42
)
model3 = HistGradientBoostingRegressor(
    max_depth=5,
    max_iter=best3[0][0],
    learning_rate=best3[0][1],
    min_samples_leaf=best3[0][2],
    l2_regularization=best3[0][3],
    early_stopping=False,
    random_state=42
)
model1.fit(X_train, y_train1)
pred1 = model1.predict(X_val)
print(f'Clicks: r2 score = {best1[1]}, mean absolute error = {mean_absolute_error(y_val1, pred1)}')
model2.fit(X_train, y_train2)
pred2 = model2.predict(X_val)
print(f'Impressions: r2 score = {best2[1]}, mean absolute error = {mean_absolute_error(y_val2, pred2)}')
model3.fit(X_train, y_train3)
pred3 = model3.predict(X_val)
print(f'Position: r2 score = {best3[1]}, mean absolute error = {mean_absolute_error(y_val3, pred3)}')

Clicks: r2 score = 0.8835630020877395, mean absolute error = 2.5040178450534047
Impressions: r2 score = 0.8125844352079867, mean absolute error = 686.9619234343456
Position: r2 score = 0.8490259057310643, mean absolute error = 3.451586686627926


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# It is grouped by content_hash_id. It ensures that data from one webpage will all go to only one split (train, val, or test).
# It is time aware: 
# It includes 90 day windows where data on every day in the window is available so that missing data will not affect the model.
# It ensures that all the data comes after the last update date to guarantee the page is not updated in the middle of the window.
# It is honest because the input does not leak future data (days 61-90 of the window), it only includes past data (days 1-60 or before).
# It includes clicks, impressions, and average position for days 1-15, 16-30, 31-45, and 46-60 for features and 61-90 for targets.
# It includes content_created_date, content_updated_date, word_count, and main_intent for features.
# These won't leak future data since all windows in the data come after update date, and word_count and main_intent won't change unless updated.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [513]:
# I am using two baselines: 
# 1. Values from the last 15 days, it will be doubled for clicks and impressions.
# 2. The average values from the 60 days, it will be doubled for clicks and impressions and weighted by impressions for position.
recent_baseline1 = X_test[:, 3] * 2
print("Clicks recent 15 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test1, recent_baseline1))
print("R2 score:", r2_score(y_test1, recent_baseline1))
mean_60d_baseline1 = np.mean(X_test[:, :4], axis=1) * 2
print("\nClicks mean 60 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test1, mean_60d_baseline1))
print("R2 score:", r2_score(y_test1, mean_60d_baseline1))
model_pred1 = model1.predict(X_test)
print("\nClicks model performance:")
print("Mean absolute error:", mean_absolute_error(y_test1, model_pred1))
print("R2 score:", r2_score(y_test1, model_pred1))
recent_baseline2 = X_test[:, 7] * 2
print("\nImpressions recent 15 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test2, recent_baseline2))
print("R2 score:", r2_score(y_test2, recent_baseline2))
mean_60d_baseline2 = np.mean(X_test[:, 4:8], axis=1) * 2
print("\nImpressions mean 60 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test2, mean_60d_baseline2))
print("R2 score:", r2_score(y_test2, mean_60d_baseline2))
model_pred2 = model2.predict(X_test)
print("\nImpressions model performance:")
print("Mean absolute error:", mean_absolute_error(y_test2, model_pred2))
print("R2 score:", r2_score(y_test2, model_pred2))
recent_baseline3 = X_test[:, 11]
print("\nPosition recent 15 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test3, recent_baseline3))
print("R2 score:", r2_score(y_test3, recent_baseline3))
mean_60d_baseline3 = (np.sum(X_test[:, 4:8] * X_test[:, 8:12], axis=1) / np.sum(X_test[:, 4:8], axis=1))
print("\nPosition mean 60 day baseline:")
print("Mean absolute error:", mean_absolute_error(y_test3, mean_60d_baseline3))
print("R2 score:", r2_score(y_test3, mean_60d_baseline3))
model_pred3 = model3.predict(X_test)
print("\nPosition model performance:")
print("Mean absolute error:", mean_absolute_error(y_test3, model_pred3))
print("R2 score:", r2_score(y_test3, model_pred3))

Clicks recent 15 day baseline:
Mean absolute error: 2.8983957219251337
R2 score: 0.8162570699245038

Clicks mean 60 day baseline:
Mean absolute error: 2.825668449197861
R2 score: 0.7866514920037863

Clicks model performance:
Mean absolute error: 2.4021781561785294
R2 score: 0.8863866603138656

Impressions recent 15 day baseline:
Mean absolute error: 743.9522281639929
R2 score: 0.7633164192420292

Impressions mean 60 day baseline:
Mean absolute error: 1090.2247771836007
R2 score: 0.7057075017721606

Impressions model performance:
Mean absolute error: 603.9551392278299
R2 score: 0.8734313180224106

Position recent 15 day baseline:
Mean absolute error: 4.92981701679662
R2 score: 0.7336817334283864

Position mean 60 day baseline:
Mean absolute error: 6.073677285624553
R2 score: 0.6226871089824088

Position model performance:
Mean absolute error: 3.759330180284617
R2 score: 0.8483991611488413


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [516]:
models = {"Clicks": (model1, y_val1),"Impressions": (model2, y_val2),"Position": (model3, y_val3)}
for name, (model, y) in models.items():
    y = np.asarray(y, dtype=float)
    pred = np.asarray(model.predict(X_val), dtype=float)
    q25, q50, q75 = np.percentile(y, [25, 50, 75])
    ranges = [("Bottom 25%", y <= q25), ("25-50%", (y > q25) & (y <= q50)), ("50-75%", (y > q50) & (y <= q75)), ("Top 25%", y > q75)]
    print(name)
    for label, mask in ranges:
        actual_mean = np.mean(y[mask])
        predicted_mean = np.mean(pred[mask])
        bias = predicted_mean - actual_mean
        print(f"{label:12} | N={np.sum(mask):4d} | Actual={actual_mean:10.2f} | Predicted = {predicted_mean:10.2f} | Bias = {bias:+10.2f}")
    print()
# It tends to overestimate smaller values and underestimate larger values for each clicks, impressions, and positions.

Clicks
Bottom 25%   | N= 755 | Actual=      0.00 | Predicted =       1.36 | Bias =      +1.36
25-50%       | N= 875 | Actual=      1.39 | Predicted =       2.10 | Bias =      +0.71
50-75%       | N= 500 | Actual=      3.81 | Predicted =       3.72 | Bias =      -0.09
Top 25%      | N= 674 | Actual=     21.13 | Predicted =      19.40 | Bias =      -1.73

Impressions
Bottom 25%   | N= 701 | Actual=    267.97 | Predicted =     665.56 | Bias =    +397.58
25-50%       | N= 701 | Actual=    540.95 | Predicted =     855.55 | Bias =    +314.60
50-75%       | N= 701 | Actual=   1001.61 | Predicted =    1140.32 | Bias =    +138.71
Top 25%      | N= 701 | Actual=   6252.26 | Predicted =    5442.40 | Bias =    -809.86

Position
Bottom 25%   | N= 701 | Actual=      5.59 | Predicted =       6.30 | Bias =      +0.71
25-50%       | N= 701 | Actual=     11.52 | Predicted =      13.32 | Bias =      +1.80
50-75%       | N= 701 | Actual=     22.14 | Predicted =      22.40 | Bias =      +0.26
Top 25%      

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.